> **TrustBreast — Notebook 3.** Tables 9, 11, 13; Figs 9–12. Saves `O3_all_patients.csv` (needed by notebook 4).

# File 3 — FIXED version (Objective 3: MC Dropout + Conformal)

## How to run
1. Colab → **Runtime → Change runtime type → CPU** (default; no GPU)
2. **Runtime → Run all**; click **Allow** when prompted for Drive access
3. Runtime: approximately **5–10 minutes**
4. Share the output of **STEP 8 — RESULTS SUMMARY**. The final cell downloads a zip (figures + CSVs) — share that as well. The zip contains `O3_all_patients.csv`, which File 4 (DiCE) requires.

## What was fixed
- The old conformal setup (which used the test set itself for calibration, 80 patients) and the old 80-patient cross-tab have been **removed**. Only the clean version (calibration = 91 held-out validation patients) is kept.
- New cell: marginal and Mondrian sets for Patients 16/87, and a corrected empty-set cross-tab (114 patients).
- New cell: split-conformal at α = 0.10 (requested by reviewer B3).
- Fixed outdated, incorrect labels in the summary and saved JSON ("455 calibration patients", "meets the 95% target", "First … on WBCD").
- Final reproducibility check: every number is matched against the previous run ✅/❌.

In [ ]:
# Colab: clone the repo (it contains the models/ folder). In local Jupyter this cell does nothing.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


## STEP 1 — Determinism + install + LOAD

In [ ]:
# ============================================
# CELL 0 — DETERMINISM  (run FIRST, before any imports)
# Adds 3 settings that make the DNN give the SAME result on every run:
#   1. os.environ flags   -> GPU/cuDNN deterministic (must be set BEFORE imports)
#   2. all seeds          -> python / numpy / tensorflow
#   3. enable_op_determinism() -> GPU floating-point order LOCK (this was the actual missing piece)
# The reseed() helper later re-fixes the RNG right before the DNN runs.
# ============================================
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("enable_op_determinism() ON  ->  DNN now reproducible")
except Exception as e:
    print("Note: enable_op_determinism unavailable (older TF). The remaining fixes still apply.")

def reseed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

print("TF:", tf.__version__, "| Determinism setup done. Now run the remaining cells.")


In [ ]:
!pip -q install scikit-learn xgboost imbalanced-learn tensorflow scipy shap lime dice-ml anthropic matplotlib seaborn

In [ ]:
# ===== LOAD the locked 99.12% model (run this FIRST) =====
# Requires the saved model folder in Google Drive: MyDrive/TrustBreast_locked/
# (produced once by File 1 - Objective 1). No retraining here, so the number is always 99.12%.
import os, pickle, joblib, numpy as np, tensorflow as tf
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
# Locked model: first the GitHub repo's models/ folder, otherwise Google Drive
SAVE_DIR = next((p for p in ['models/TrustBreast_locked', '../models/TrustBreast_locked']
                 if os.path.isdir(p)), None)
if SAVE_DIR is None:
    SAVE_DIR = '/content/drive/MyDrive/TrustBreast_locked'
    from google.colab import drive; drive.mount('/content/drive')
print('Loading locked model from:', SAVE_DIR)
rf_model  = joblib.load(SAVE_DIR + '/rf_model.pkl')
xgb_model = joblib.load(SAVE_DIR + '/xgb_model.pkl')
scaler    = joblib.load(SAVE_DIR + '/scaler.pkl')
dnn_best  = tf.keras.models.load_model(SAVE_DIR + '/dnn_best.keras')
dnn_model = tf.keras.models.load_model(SAVE_DIR + '/dnn_model.keras')
with open(SAVE_DIR + '/state.pkl','rb') as f: state = pickle.load(f)
globals().update({k:v for k,v in state.items() if v is not None})
if globals().get('prob_ensemble_val') is None and 'X_val_sc' in globals():
    _rf=rf_model.predict_proba(X_val_sc)[:,1]; _xg=xgb_model.predict_proba(X_val_sc)[:,1]
    _dn=dnn_best.predict(X_val_sc, verbose=0).ravel(); prob_ensemble_val=(_rf+_xg+_dn)/3
model_dnn=dnn_best; rf_aug=rf_model; xgb_aug=xgb_model; feature_names=list(X.columns)
print('LOADED locked model. Ensemble accuracy:', round(accuracy_score(y_test, ens_pred)*100,2), 'percent')

In [ ]:
# --- aliases so O2/O3/O4 cells find the trained models ---
model_dnn = dnn_best        # O3 (MC Dropout) expects this name
rf_aug    = rf_model        # O4 (DiCE) expects this name
xgb_aug   = xgb_model       # O4 (DiCE) expects this name
feature_names = list(X.columns)
print('Bridge ready. Ensemble threshold from O1:', best_ens_thr)

In [ ]:
import numpy as np
y_test_arr = np.asarray(y_test).ravel().astype(int)
print('Test labels:', np.bincount(y_test_arr), '(0=B, 1=M)')

## STEP 2 — Test set scale + reproducible MC-Dropout forward pass (stateless masks)

In [ ]:
import numpy as np
import tensorflow as tf

# ============================================================
# STEP 3.1 — MinMaxScaler Fix
# ============================================================
print("=" * 55)
print("STEP 3.1 — MinMaxScaler APPLY")
print("=" * 55)

# MinMaxScaler has different attributes (no mean_)
print(f"Scaler type: MinMaxScaler")
print(f"Feature range: {scaler.feature_range}")
print(f"Data min (first 3): {scaler.data_min_[:3].round(3)}")
print(f"Data max (first 3): {scaler.data_max_[:3].round(3)}")

# ── SCALE X_test ─────────────────────────────────────────────
X_test_scaled = scaler.transform(
    np.array(X_test)).astype('float32')

print(f"\nBefore → Min: {np.array(X_test).min():.2f}, "
      f"Max: {np.array(X_test).max():.2f}")
print(f"After  → Min: {X_test_scaled.min():.4f}, "
      f"Max: {X_test_scaled.max():.4f}")
print(f"         (Should be 0.0 to 1.0 — MinMaxScaler range)")

# ── DNN PREDICTIONS VERIFY ───────────────────────────────────
print("\n── DNN predictions after scaling ──")
preds_scaled = model_dnn.predict(
    X_test_scaled, verbose=0).flatten()

print(f"Near 1.0 (>0.99):    {np.sum(preds_scaled > 0.99)}")
print(f"Near 0.0 (<0.01):    {np.sum(preds_scaled < 0.01)}")
print(f"Borderline (0.3-0.7):{np.sum((preds_scaled>0.3)&(preds_scaled<0.7))}")
print(f"Min: {preds_scaled.min():.4f} | "
      f"Max: {preds_scaled.max():.4f} | "
      f"Mean: {preds_scaled.mean():.4f}")

preds_class = (preds_scaled > 0.5).astype(int)
acc = np.mean(preds_class == y_test_arr)
print(f"\nAccuracy on scaled X_test: {acc*100:.2f}%")

# ── MC DROPOUT VERIFY ON SCALED DATA ─────────────────────────
# Define mc_forward_pass here to ensure it's available
def mc_forward_pass(x, pass_id=0):
    """
    Fully reproducible MC Dropout forward pass.
    The dropout mask comes from a fixed formula (stateless RNG), so every
    run gives EXACTLY the same result, regardless of what code ran before.
    BatchNorm stays in inference mode (stored statistics).
    """
    out = x
    d = 0
    for layer in model_dnn.layers:
        if isinstance(layer, tf.keras.layers.Dropout):
            rate = float(layer.rate)
            keep = tf.random.stateless_uniform(tf.shape(out), seed=[1000 + pass_id, d]) >= rate
            out  = tf.where(keep, out / (1.0 - rate), tf.zeros_like(out))
            d += 1
        elif isinstance(layer, tf.keras.layers.BatchNormalization):
            out = layer(out, training=False)
        else:
            out = layer(out)
    return out
print("\n── MC Dropout verify (scaled data) ──")
print("10 passes on first 5 patients:")

for pat_idx in range(5):
    sample = tf.convert_to_tensor(
        X_test_scaled[pat_idx:pat_idx+1])
    passes = [
        mc_forward_pass(sample, k).numpy()[0][0]
        for k in range(10)
    ]
    mean_p = np.mean(passes)
    std_p  = np.std(passes)
    true_l = 'M' if y_test_arr[pat_idx]==1 else 'B'
    print(f"  Patient {pat_idx} [{true_l}]: "
          f"mean={mean_p:.4f}, std={std_p:.4f}  "
          f"passes={[round(p,3) for p in passes[:4]]}...")

# ── X_test_arr UPDATE ────────────────────────────────────────
X_test_arr = X_test_scaled
print(f"\n✅ X_test_arr updated — shape: {X_test_arr.shape}")
print("✅ Step 3.1 TRULY complete!")
print("   Next: Step 3.2 — 100 full MC passes!")

## STEP 3 — 100 MC-Dropout passes (Table 9)

In [ ]:
# ============================================================
# STEP 3.2 — 100 Stochastic Forward Passes (Full Test Set)
# ============================================================
import numpy as np
import tensorflow as tf
from tqdm import tqdm

print("=" * 55)
print("STEP 3.2 — 100 MC DROPOUT PASSES")
print("=" * 55)
print(f"Test patients: {len(X_test_arr)}")
print(f"Passes per patient: 100")
print(f"Total forward passes: {len(X_test_arr) * 100}")
print()

N_PASSES = 100
n_patients = len(X_test_arr)

# ── 1. RUN ALL PASSES ────────────────────────────────────────
# Shape: (100, 114) — 100 passes, 114 patients
mc_probs = np.zeros((N_PASSES, n_patients), dtype=np.float32)

# ── SEED (reproducibility: locks MC-Dropout run-to-run & across O3/O4) ──
np.random.seed(42)
tf.random.set_seed(42)

print("Running MC passes...")
for i in tqdm(range(N_PASSES), desc="MC Passes"):
    tf.random.set_seed(1000 + i)
    batch = tf.convert_to_tensor(X_test_arr)
    preds = mc_forward_pass(batch, i).numpy().flatten()
    mc_probs[i] = preds

print(f"\n✅ mc_probs shape: {mc_probs.shape}")
print(f"   (rows=passes, cols=patients)")

# ── 2. STEP 3.3 PREVIEW: MEAN + STD ─────────────────────────
# (Full 3.3 is a separate step — verify only here)
mc_mean = mc_probs.mean(axis=0)   # shape: (114,)
mc_std  = mc_probs.std(axis=0)    # shape: (114,)

print(f"\n── Quick stats (Step 3.3 preview) ──")
print(f"Mean predictions:")
print(f"  Min: {mc_mean.min():.4f} | "
      f"Max: {mc_mean.max():.4f} | "
      f"Avg: {mc_mean.mean():.4f}")
print(f"\nUncertainty (std):")
print(f"  Min: {mc_std.min():.4f}")
print(f"  Max: {mc_std.max():.4f}")
print(f"  Mean: {mc_std.mean():.4f}")
print(f"  High uncertainty (std>0.10): "
      f"{np.sum(mc_std > 0.10)} patients")
print(f"  Very high (std>0.15):        "
      f"{np.sum(mc_std > 0.15)} patients")

# Top 5 uncertain patients
top5_idx = np.argsort(mc_std)[::-1][:5]
print(f"\nTop 5 most uncertain patients:")
print(f"  {'Idx':>4} {'True':>6} {'Mean':>7} "
      f"{'Std':>7} {'Flag':>8}")
print(f"  {'-'*38}")
for idx in top5_idx:
    true_l = 'M' if y_test_arr[idx]==1 else 'B'
    flag = "⚠ BIOPSY" if mc_std[idx] > 0.15 else ""
    print(f"  {idx:>4} {true_l:>6} "
          f"{mc_mean[idx]:>7.4f} "
          f"{mc_std[idx]:>7.4f} {flag}")

print()
print("=" * 55)
print("STEP 3.2 COMPLETE ✅")
print("=" * 55)
print(f"mc_probs saved — shape {mc_probs.shape}")
print("Next: Step 3.3 — Mean + Std + 95% CI compute!")

## STEP 4 — Mean, std, 95% interval + Fig 9

In [ ]:
# ============================================================
# STEP 3.3 — Mean + Std + 95% Confidence Interval
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("=" * 55)
print("STEP 3.3 — MEAN + STD + 95% CI")
print("=" * 55)

# ── 1. CORE STATISTICS COMPUTE ───────────────────────────────
mc_mean = mc_probs.mean(axis=0)          # (114,)
mc_std  = mc_probs.std(axis=0)           # (114,)

# 95% CI — percentile method (distribution-free, robust)
mc_ci_lower = np.percentile(mc_probs, 2.5, axis=0)   # (114,)
mc_ci_upper = np.percentile(mc_probs, 97.5, axis=0)  # (114,)
mc_ci_width = mc_ci_upper - mc_ci_lower               # (114,)

# Final prediction from mean
mc_pred_class = (mc_mean > 0.5).astype(int)

# Accuracy
acc_mc = np.mean(mc_pred_class == y_test_arr)

print(f"MC Dropout Statistics (100 passes, 114 patients):")
print(f"  Accuracy:          {acc_mc*100:.2f}%")
print(f"  Mean std:          {mc_std.mean():.4f}")
print(f"  Mean CI width:     {mc_ci_width.mean():.4f}")
print(f"  Max uncertainty:   {mc_std.max():.4f} "
      f"(patient {mc_std.argmax()})")

# ── 2. UNCERTAINTY CATEGORIES ────────────────────────────────
# Clinical triage thresholds
low_unc    = mc_std <= 0.05
medium_unc = (mc_std > 0.05) & (mc_std <= 0.15)
high_unc   = mc_std > 0.15

print(f"\nUncertainty triage:")
print(f"  Low    (std ≤ 0.05): {low_unc.sum():3d} patients — confident")
print(f"  Medium (0.05-0.15):  {medium_unc.sum():3d} patients — review")
print(f"  High   (std > 0.15): {high_unc.sum():3d} patients — ⚠ BIOPSY")

# ── 3. RESULTS DATAFRAME ─────────────────────────────────────
df_mc = pd.DataFrame({
    'patient_idx':  np.arange(len(y_test_arr)),
    'true_label':   ['M' if y==1 else 'B' for y in y_test_arr],
    'mc_mean':      mc_mean.round(4),
    'mc_std':       mc_std.round(4),
    'ci_lower':     mc_ci_lower.round(4),
    'ci_upper':     mc_ci_upper.round(4),
    'ci_width':     mc_ci_width.round(4),
    'pred_class':   ['M' if p==1 else 'B' for p in mc_pred_class],
    'correct':      mc_pred_class == y_test_arr,
    'uncertainty':  ['High' if h else ('Medium' if m else 'Low')
                     for h,m in zip(high_unc, medium_unc)],
    'biopsy_flag':  high_unc
})

print(f"\nTop 10 most uncertain patients:")
print(df_mc.sort_values('mc_std', ascending=False)
      [['patient_idx','true_label','mc_mean',
        'mc_std','ci_lower','ci_upper','uncertainty']]
      .head(10).to_string(index=False))

# ── 4. MISCLASSIFIED + UNCERTAIN ─────────────────────────────
missed = df_mc[~df_mc['correct']]
print(f"\nMisclassified patients: {len(missed)}")
if len(missed) > 0:
    print(missed[['patient_idx','true_label','pred_class',
                  'mc_mean','mc_std','uncertainty']]
          .to_string(index=False))

# ── 5. VISUALIZATION ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Uncertainty distribution ─────────────────────────
axes[0].hist(mc_std, bins=25, color='steelblue',
             edgecolor='white', linewidth=0.5)
axes[0].axvline(0.05, color='orange', linestyle='--',
                linewidth=1.5, label='Medium threshold (0.05)')
axes[0].axvline(0.15, color='red', linestyle='--',
                linewidth=1.5, label='Biopsy threshold (0.15)')
axes[0].set_xlabel('MC Dropout Std (Uncertainty)')
axes[0].set_ylabel('Number of Patients')
axes[0].set_title('Uncertainty Distribution\n(100 MC passes)',
                  fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# ── Plot 2: Mean prediction + 95% CI (sorted) ────────────────
sort_idx = np.argsort(mc_mean)
colors_p = ['#E74C3C' if y==1 else '#3498DB'
            for y in y_test_arr[sort_idx]]

axes[1].scatter(range(len(sort_idx)), mc_mean[sort_idx],
                c=colors_p, s=18, zorder=3, alpha=0.8)
axes[1].fill_between(
    range(len(sort_idx)),
    mc_ci_lower[sort_idx],
    mc_ci_upper[sort_idx],
    alpha=0.2, color='gray', label='95% CI')
axes[1].axhline(0.5, color='black', linestyle='--',
                linewidth=1, alpha=0.5, label='Decision boundary')
axes[1].set_xlabel('Patients (sorted by mean prediction)')
axes[1].set_ylabel('Malignancy Probability')
axes[1].set_title('MC Mean ± 95% CI\nRed=Malignant, Blue=Benign',
                  fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# ── Plot 3: Std vs Mean (uncertainty landscape) ──────────────
scatter_colors = ['#E74C3C' if y==1 else '#3498DB'
                  for y in y_test_arr]
sc = axes[2].scatter(mc_mean, mc_std,
                     c=scatter_colors, s=25,
                     alpha=0.75, zorder=3)
axes[2].axhline(0.15, color='red', linestyle='--',
                linewidth=1.5, label='Biopsy threshold (0.15)')
axes[2].axhline(0.05, color='orange', linestyle='--',
                linewidth=1.5, label='Medium threshold (0.05)')
axes[2].axvline(0.5, color='gray', linestyle=':',
                linewidth=1, alpha=0.7)

# Annotate top uncertain patients
for idx in np.argsort(mc_std)[::-1][:5]:
    axes[2].annotate(
        f"P{idx}",
        (mc_mean[idx], mc_std[idx]),
        textcoords="offset points",
        xytext=(5, 3), fontsize=7, color='darkred')

red_patch   = mpatches.Patch(color='#E74C3C', label='Malignant')
blue_patch  = mpatches.Patch(color='#3498DB', label='Benign')
axes[2].legend(handles=[red_patch, blue_patch] +
               axes[2].get_legend_handles_labels()[0][0:2],
               fontsize=7)
axes[2].set_xlabel('MC Mean (Malignancy Probability)')
axes[2].set_ylabel('MC Std (Uncertainty)')
axes[2].set_title('Uncertainty Landscape\nStd vs Mean prediction',
                  fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle(
    f'MC Dropout Uncertainty — 100 passes, 114 patients\n'
    f'Accuracy: {acc_mc*100:.2f}% | '
    f'High uncertainty: {high_unc.sum()} patients flagged',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mc_dropout_uncertainty.png', dpi=150,
            bbox_inches='tight')
plt.show()

# ── 6. PAPER PARAGRAPH ───────────────────────────────────────
print()
print("=" * 55)
print("PAPER PARAGRAPH — Results section")
print("=" * 55)
print(f"""
MC Dropout uncertainty quantification was performed
using 100 stochastic forward passes per patient
(n={len(y_test_arr)} test patients).

Key findings:
  - Accuracy (MC mean):     {acc_mc*100:.2f}%
  - Mean uncertainty (std): {mc_std.mean():.4f}
  - Mean 95% CI width:      {mc_ci_width.mean():.4f}

Uncertainty triage:
  - Low    (std ≤ 0.05): {low_unc.sum()} patients ({low_unc.mean()*100:.1f}%)
  - Medium (0.05-0.15):  {medium_unc.sum()} patients ({medium_unc.mean()*100:.1f}%)
  - High   (std > 0.15): {high_unc.sum()} patients ({high_unc.mean()*100:.1f}%)
    → Flagged for biopsy review

Most uncertain: Patient {mc_std.argmax()}
  mean={mc_mean[mc_std.argmax()]:.4f},
  std={mc_std.max():.4f},
  95% CI [{mc_ci_lower[mc_std.argmax()]:.4f},
           {mc_ci_upper[mc_std.argmax()]:.4f}]
""")

print("=" * 55)
print("STEP 3.3 COMPLETE ✅")
print("=" * 55)
print("Saved: mc_dropout_uncertainty.png")
print("Variables ready: mc_mean, mc_std, mc_ci_lower,")
print("                 mc_ci_upper, df_mc")
print("Next: Step 3.4 — Flag high-uncertainty cases table!")

## STEP 5 — Patients 16 and 87 (ensemble vs DNN)

In [ ]:
import numpy as np

y_arr    = np.asarray(y_test).ravel().astype(int)
prob_arr = np.asarray(prob_ensemble).ravel()
pred_arr = np.asarray(ens_pred).ravel().astype(int)
std_rank = {int(ix): r + 1 for r, ix in enumerate(np.argsort(-mc_std))}

print("=" * 66)
print("  ENSEMBLE ERRORS")
print("=" * 66)
fn = np.where((y_arr == 1) & (pred_arr == 0))[0]
fp = np.where((y_arr == 0) & (pred_arr == 1))[0]
print(f"  False negatives : {fn.tolist()}")
print(f"  False positives : {fp.tolist()}")

print("\n" + "=" * 66)
print("  PER-PATIENT RECORD")
print("=" * 66)
for i in sorted(set([16, 87] + fn.tolist() + fp.tolist())):
    corr = "correct" if pred_arr[i] == y_arr[i] else "WRONG"
    print(f"\n  --- Patient {i} ---")
    print(f"    true label       : {'M' if y_arr[i] else 'B'}")
    print(f"    ensemble prob    : {prob_arr[i]:.4f}   (threshold 0.35)")
    print(f"    ensemble pred    : {'M' if pred_arr[i] else 'B'}   -> {corr}")
    print(f"    DNN MC mean      : {mc_mean[i]:.4f}   -> {'M' if mc_mean[i] >= 0.5 else 'B'}")
    print(f"    DNN MC std       : {mc_std[i]:.4f}   (rank {std_rank[i]}/114)")
    print(f"    flagged (>0.15)  : {bool(mc_std[i] > 0.15)}")
    print(f"    95% interval     : [{mc_ci_lower[i]:.3f}, {mc_ci_upper[i]:.3f}]")

print("\n" + "=" * 66)
print("  HIGH-UNCERTAINTY LIST")
print("=" * 66)
hi = np.where(mc_std > 0.15)[0]
print(f"  Total flagged : {len(hi)}")
print(f"  Indices       : {sorted(hi.tolist())}")
for i in sorted(set(fn.tolist() + fp.tolist())):
    print(f"  Patient {i} (ensemble error) flagged? -> {i in hi.tolist()}")

print("\n" + "=" * 66)
print("  MC-MEAN ERRORS (separate from the ensemble)")
print("=" * 66)
mc_pred = (mc_mean >= 0.5).astype(int)
mc_err  = np.where(mc_pred != y_arr)[0]
print(f"  MC-mean accuracy : {(mc_pred == y_arr).mean()*100:.2f}%")
print(f"  MC-mean errors   : {mc_err.tolist()}")


## STEP 6 — ★ Clean split-conformal (calibration = 91 validation patients) + Mondrian + ECE (Table 11, Fig 10)

In [ ]:
# ============================================================
#  OBJECTIVE 3 — CONFORMAL (clean split) + MONDRIAN + ECE
#  REPRODUCIBLE, LEAKAGE-FREE REPLACEMENT
#
#  This cell replaces the OLD File 3 conformal cells (Setup / Fix /
#  empty-set / reliability) — DELETE all of them and
#  keep only this one cell.
#
#  What was fixed (reviewer points):
#   1. Calibration now on X_VAL (91 real patients the model
#      did NOT see in training) -> exchangeability preserved
#   2. Empty-set padding REMOVED -> coverage is not inflated
#   3. MONDRIAN (class-conditional): SEPARATE benign/malignant
#      tau -> coverage of the malignant (dangerous) class reported separately
#   4. Seed-locked -> SAME numbers on every run (reproducible)
#   5. Reliability figure: "flagged for additional review"
#      language, no "MANDATORY BIOPSY"
#
#  Required variables (from File 1 LOAD + O3 MC cells):
#   model_dnn, mc_forward_pass, X_test_arr (scaled test),
#   y_test, X_val_sc, y_val, mc_mean, mc_std
# ============================================================
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# ---- 0. REPRODUCIBILITY: seed lock ----
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

ALPHA = 0.05                       # 95% target coverage
N_CAL_PASSES = 100                 # MC passes on calibration set (same as test)

# ---- 1. Safety: are the required variables present? ----
assert 'X_val_sc' in globals() and 'y_val' in globals(), \
    "X_val_sc/y_val not found — run the File 1 LOAD cell first (state.pkl restores them)."
assert 'X_test_arr' in globals(), \
    "X_test_arr not found — run the O3 MC Dropout cells first."
assert 'mc_mean' in globals(), \
    "mc_mean not found — run the 100-pass MC Dropout cell first."

y_val_arr  = np.array(y_val).astype(int).ravel()
y_test_arr = np.array(y_test).astype(int).ravel()
X_cal      = np.array(X_val_sc).astype('float32')
n_cal      = len(X_cal)
print("="*60)
print("CLEAN SPLIT-CONFORMAL  (calibration = X_val, held-out)")
print("="*60)
print(f"Calibration patients (X_val): {n_cal}  "
      f"[B:{int((y_val_arr==0).sum())}  M:{int((y_val_arr==1).sum())}]")
print(f"Test patients:                {len(y_test_arr)}")
print(f"Alpha={ALPHA}  ->  target coverage {int((1-ALPHA)*100)}%")

# ---- 2. MC-mean probability on CALIBRATION set (seed-locked) ----
# mc_forward_pass: Dropout training=True, BatchNorm training=False (BatchNorm-safe)
tf.random.set_seed(SEED)
cal_probs = np.zeros((N_CAL_PASSES, n_cal), dtype=np.float32)
xb = tf.convert_to_tensor(X_cal)
for i in range(N_CAL_PASSES):
    cal_probs[i] = mc_forward_pass(xb, 5000 + i).numpy().ravel()
cal_mean = cal_probs.mean(axis=0)          # P(malignant) per calibration patient

# test MC-mean already computed as mc_mean (P malignant)
test_mean = np.asarray(mc_mean).ravel()

# ---- 3. Nonconformity score = 1 - p(true class) ----
# benign (y=0):     true prob = 1 - p_malignant
# malignant (y=1):  true prob = p_malignant
cal_scores = np.where(y_val_arr == 1, 1.0 - cal_mean, cal_mean)

def conformal_tau(scores, alpha=ALPHA):
    """Finite-sample adjusted quantile (Angelopoulos & Bates, 2021)."""
    n = len(scores)
    q = np.ceil((1 - alpha) * (n + 1)) / n
    q = min(q, 1.0)
    return float(np.quantile(scores, q)), float(q)

# ---- 4a. MARGINAL conformal (single tau) ----
tau, q_level = conformal_tau(cal_scores, ALPHA)
print("\n" + "-"*60)
print("MARGINAL conformal")
print("-"*60)
print(f"  q_level = ceil((1-a)(n+1))/n = {q_level:.4f}  "
      f"({int(np.ceil((1-ALPHA)*(n_cal+1)))}/{n_cal})")
print(f"  tau     = {tau:.4f}")

# Build prediction sets on TEST (score <= tau => class included). NO empty-set padding.
score_mal_test = 1.0 - test_mean     # score for label M
score_ben_test = test_mean           # score for label B
inc_ben = score_ben_test <= tau
inc_mal = score_mal_test <= tau

pred_sets, covered = [], []
singleton = both = empty = 0
for i in range(len(y_test_arr)):
    s = []
    if inc_ben[i]: s.append(0)
    if inc_mal[i]: s.append(1)
    pred_sets.append(s)
    if len(s) == 2: both += 1
    elif len(s) == 1: singleton += 1
    else: empty += 1
    covered.append(y_test_arr[i] in s)      # empty set => NOT covered (honest)

covered = np.array(covered)
marginal_cov = covered.mean()
print(f"  Empirical coverage: {marginal_cov*100:.2f}%   "
      f"(singleton {singleton} | ambiguous{{B,M}} {both} | empty {empty})")
print(f"  Empty sets are NOT padded — reported coverage is the true conformal coverage.")

# ---- 4b. MONDRIAN (class-conditional) conformal ----
# Separate tau per class -> prevents the malignant (clinically dangerous) class's
# coverage from hiding behind the marginal coverage.
cal_scores_ben = cal_scores[y_val_arr == 0]
cal_scores_mal = cal_scores[y_val_arr == 1]
tau_ben, qb = conformal_tau(cal_scores_ben, ALPHA)
tau_mal, qm = conformal_tau(cal_scores_mal, ALPHA)

# Mondrian sets: benign-label included if its score <= tau_ben; malignant-label if <= tau_mal
inc_ben_m = score_ben_test <= tau_ben
inc_mal_m = score_mal_test <= tau_mal
cov_ben_hits = cov_mal_hits = 0
n_ben = int((y_test_arr == 0).sum()); n_mal = int((y_test_arr == 1).sum())
pred_sets_m = []
for i in range(len(y_test_arr)):
    s = []
    if inc_ben_m[i]: s.append(0)
    if inc_mal_m[i]: s.append(1)
    pred_sets_m.append(s)
    if y_test_arr[i] == 0 and 0 in s: cov_ben_hits += 1
    if y_test_arr[i] == 1 and 1 in s: cov_mal_hits += 1
cov_ben = cov_ben_hits / max(n_ben, 1)
cov_mal = cov_mal_hits / max(n_mal, 1)

print("\n" + "-"*60)
print("MONDRIAN (class-conditional) conformal")
print("-"*60)
print(f"  tau_benign    = {tau_ben:.4f}  ->  benign-class coverage    = {cov_ben*100:.2f}%  (n={n_ben})")
print(f"  tau_malignant = {tau_mal:.4f}  ->  malignant-class coverage = {cov_mal*100:.2f}%  (n={n_mal})")
print(f"  (Malignant coverage guaranteed separately — not hidden behind the marginal)")

# ---- 5. ECE / MCE (reliability) on TEST, using MC-mean confidence ----
def compute_ece(probs, labels, n_bins=10):
    """probs = P(malignant); confidence = max(p, 1-p); ECE + MCE + per-bin."""
    conf = np.maximum(probs, 1 - probs)
    pred = (probs >= 0.5).astype(int)
    correct = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = mce = 0.0
    xs_conf, xs_acc, xs_n = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            xs_conf.append((lo+hi)/2); xs_acc.append(0.0); xs_n.append(0); continue
        acc_b = correct[m].mean(); conf_b = conf[m].mean(); w = m.mean()
        gap = abs(acc_b - conf_b)
        ece += w * gap
        mce = max(mce, gap)
        xs_conf.append(conf_b); xs_acc.append(acc_b); xs_n.append(int(m.sum()))
    return ece, mce, np.array(xs_conf), np.array(xs_acc), np.array(xs_n)

ECE, MCE, bin_conf, bin_acc, bin_n = compute_ece(test_mean, y_test_arr, n_bins=10)
calib_tag = "well calibrated" if ECE < 0.05 else ("acceptable" if ECE < 0.10 else "needs recalibration")
print("\n" + "-"*60)
print("CALIBRATION")
print("-"*60)
print(f"  ECE = {ECE:.4f}   MCE = {MCE:.4f}   ->  {calib_tag}")

# ---- 6. FIGURE 4.11 — Reliability diagram (clean language) ----
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))

ax[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration', linewidth=1.5)
valid = bin_n > 0
ax[0].plot(bin_conf[valid], bin_acc[valid], 'o-', color='#C0392B',
           linewidth=2, markersize=7, label='Model')
ax[0].fill_between([0, 1], [0, 1], [0, 0], color='gray', alpha=0.06)
ax[0].set_xlabel('Mean predicted confidence'); ax[0].set_ylabel('Empirical accuracy')
ax[0].set_title(f'Reliability Diagram\nECE = {ECE:.4f}  ({calib_tag})',
                fontsize=12, fontweight='bold')
ax[0].legend(loc='upper left'); ax[0].grid(alpha=0.3)
ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1)

# conformal coverage summary panel
labels = ['Marginal', 'Benign\n(Mondrian)', 'Malignant\n(Mondrian)']
covs   = [marginal_cov*100, cov_ben*100, cov_mal*100]
colors = ['#2E86C1', '#27AE60', '#C0392B']
bars = ax[1].bar(labels, covs, color=colors, edgecolor='white', width=0.6)
ax[1].axhline((1-ALPHA)*100, ls='--', color='black', linewidth=1.2,
              label=f'{int((1-ALPHA)*100)}% target')
for b, v in zip(bars, covs):
    ax[1].text(b.get_x()+b.get_width()/2, v+0.4, f'{v:.2f}%',
               ha='center', fontsize=11, fontweight='bold')
ax[1].set_ylabel('Empirical coverage (%)'); ax[1].set_ylim(80, 103)
ax[1].set_title(f'Conformal Coverage  (alpha={ALPHA})\n'
                f'tau_marginal={tau:.4f}',
                fontsize=12, fontweight='bold')
ax[1].legend(loc='lower right'); ax[1].grid(axis='y', alpha=0.3)

plt.suptitle('TrustBreast | Objective 3 — Conformal Prediction & Calibration',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('O3_reliability_conformal_FIXED.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- 7. FIGURE 4.12 — Conformal detail (nonconformity + set composition) ----
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

ax[0].hist(cal_scores, bins=25, color='#8E44AD', alpha=0.85,
           edgecolor='white', label='Calibration nonconformity')
ax[0].axvline(tau, ls='--', color='red', linewidth=2,
              label=f'tau = {tau:.4f} ({q_level:.3f} q)')
ax[0].set_xlabel('Nonconformity score  (1 - P(true class))')
ax[0].set_ylabel('Count')
ax[0].set_title(f'Calibration Scores on X_val (n={n_cal})\nHeld-out — exchangeability preserved',
                fontsize=11, fontweight='bold')
ax[0].legend(); ax[0].grid(alpha=0.3)

set_labels = ['Singleton\n(confident)', 'Ambiguous\n{B, M}', 'Empty\n(none)']
set_counts = [singleton, both, empty]
set_cols   = ['#27AE60', '#E67E22', '#95A5A6']
bars = ax[1].bar(set_labels, set_counts, color=set_cols, edgecolor='white', width=0.6)
for b, v in zip(bars, set_counts):
    pct = v/len(y_test_arr)*100
    ax[1].text(b.get_x()+b.get_width()/2, v+0.5, f'{v}\n({pct:.1f}%)',
               ha='center', fontsize=10, fontweight='bold')
ax[1].set_ylabel('Number of test patients')
ax[1].set_title('Prediction-Set Composition\nAmbiguous = flagged for additional review',
                fontsize=11, fontweight='bold')
ax[1].grid(axis='y', alpha=0.3)

# marginal vs mondrian coverage gauge-style
ax[2].bar(['Marginal', 'Benign', 'Malignant'],
          [marginal_cov*100, cov_ben*100, cov_mal*100],
          color=['#2E86C1', '#27AE60', '#C0392B'], edgecolor='white', width=0.6)
ax[2].axhline((1-ALPHA)*100, ls='--', color='black',
              label=f'{int((1-ALPHA)*100)}% target')
ax[2].set_ylim(80, 103); ax[2].set_ylabel('Coverage (%)')
ax[2].set_title('Marginal vs Class-Conditional Coverage',
                fontsize=11, fontweight='bold')
ax[2].legend(); ax[2].grid(axis='y', alpha=0.3)

plt.suptitle('TrustBreast | Conformal Prediction — Clean Split (calibration on held-out X_val)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('O3_conformal_detail_FIXED.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- 8. FINAL SUMMARY (these numbers go into the thesis) ----
print("\n" + "="*60)
print("OBJECTIVE 3 — FINAL REPRODUCIBLE NUMBERS  (use in thesis)")
print("="*60)
print(f"  Calibration set:        X_val (held-out), n={n_cal}")
print(f"  Alpha:                  {ALPHA}  (target {int((1-ALPHA)*100)}%)")
print(f"  tau (marginal):         {tau:.4f}")
print(f"  Marginal coverage:      {marginal_cov*100:.2f}%")
print(f"  tau_benign:             {tau_ben:.4f}   coverage {cov_ben*100:.2f}%")
print(f"  tau_malignant:          {tau_mal:.4f}   coverage {cov_mal*100:.2f}%")
print(f"  ECE:                    {ECE:.4f}  ({calib_tag})")
print(f"  MCE:                    {MCE:.4f}")
print(f"  Singleton / Ambiguous / Empty:  {singleton} / {both} / {empty}")
print("="*60)
print("All of this is SEED-LOCKED — re-running gives the SAME result.")
print("Now enter these numbers in the thesis text + Table 4.4/4.5.")

# expose for downstream cells
tau_fixed = tau
coverage  = marginal_cov


## STEP 6b — Patient 16 / 87 case figures (Fig 11, Fig 12)

In [ ]:
# ============================================================
#  FIGURE — Patient case studies (16 = ensemble's only error, 87 = ensemble corrected the DNN)
#  Every number comes from this run's LIVE arrays.
#  Ensemble probability and DNN MC-mean are now shown SEPARATELY.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

p_ens_all  = np.asarray(prob_ensemble).ravel()
pred_ens_all = np.asarray(ens_pred).ravel().astype(int)
y_arr      = np.asarray(y_test_arr).ravel().astype(int)

for PATIENT_IDX, OUTNAME in [(16, 'patient16_case.png'), (87, 'patient87_case.png')]:

    p_mean = float(mc_mean[PATIENT_IDX])
    p_std  = float(mc_std[PATIENT_IDX])
    p_samples = np.asarray(mc_probs)[:, PATIENT_IDX]
    ci_lo, ci_hi = float(np.percentile(p_samples, 2.5)), float(np.percentile(p_samples, 97.5))

    p_ens     = float(p_ens_all[PATIENT_IDX])
    ens_lab   = 'M' if pred_ens_all[PATIENT_IDX] == 1 else 'B'
    true_label = 'M' if y_arr[PATIENT_IDX] == 1 else 'B'
    mc_lab    = 'M' if p_mean >= 0.5 else 'B'

    conformal_labels = pred_sets[PATIENT_IDX]
    conformal_str = ('{' + ', '.join('M' if l == 1 else 'B' for l in conformal_labels) + '}') \
                    if len(conformal_labels) else '{} (empty)'
    rank_pos = int((-mc_std).argsort().tolist().index(PATIENT_IDX)) + 1

    ens_ok = (ens_lab == true_label)
    mc_ok  = (mc_lab  == true_label)

    print(f"Patient {PATIENT_IDX}: true={true_label} | ensemble={p_ens:.4f}->{ens_lab} "
          f"({'correct' if ens_ok else 'ERROR'}) | MC-mean={p_mean:.4f}->{mc_lab} "
          f"({'correct' if mc_ok else 'ERROR'}) | std={p_std:.4f} rank {rank_pos} | conformal={conformal_str}")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

    # ---- Left: MC Dropout distribution ----
    ax[0].hist(p_samples, bins=20, color='#C0392B', alpha=0.7, edgecolor='white')
    ax[0].axvline(p_mean, color='black', lw=2, label=f'MC mean = {p_mean:.3f}')
    ax[0].axvline(p_ens, color='#1F618D', lw=2, ls='-.', label=f'Ensemble = {p_ens:.3f}')
    ax[0].axvline(0.5, color='gray', ls=':', lw=1.5, label='DNN boundary (0.5)')
    ax[0].axvspan(ci_lo, ci_hi, alpha=0.12, color='purple', label=f'95% CI [{ci_lo:.2f}, {ci_hi:.2f}]')
    ax[0].set_xlabel('Predicted probability (Malignant)')
    ax[0].set_ylabel('MC sample count')
    ax[0].set_title(f'Patient {PATIENT_IDX} — DNN MC Dropout distribution\n'
                    f'MC mean={p_mean:.3f}  Std={p_std:.3f}',
                    fontsize=10, fontweight='bold')
    ax[0].legend(fontsize=8)

    # ---- Middle: uncertainty ranking ----
    order = np.argsort(-mc_std)
    pos = int(np.where(order == PATIENT_IDX)[0][0])
    colors_b = ['#C0392B' if i == pos else '#5DADE2' for i in range(len(mc_std))]
    ax[1].bar(range(len(mc_std)), mc_std[order], color=colors_b, width=1.0)
    ax[1].axhline(0.15, color='red', ls='--', lw=1.5, label='Escalation threshold (0.15)')
    ax[1].annotate(f'Patient {PATIENT_IDX}\n(rank {rank_pos})', xy=(pos, p_std),
                   xytext=(pos + 15, p_std + 0.02), fontsize=9, fontweight='bold',
                   color='#C0392B', arrowprops=dict(arrowstyle='->', color='#C0392B'))
    ax[1].set_xlabel('Test patients (sorted by uncertainty)')
    ax[1].set_ylabel('MC Dropout std')
    ax[1].set_title(f'{len(mc_std)} patients — uncertainty ranking\n'
                    f'Patient {PATIENT_IDX} at rank {rank_pos}', fontsize=10, fontweight='bold')
    ax[1].legend(fontsize=8)

    # ---- Right: per-mechanism summary (ensemble and DNN kept separate) ----
    ax[2].axis('off')
    ax[2].text(0.5, 0.94, f'Patient {PATIENT_IDX} — per-mechanism summary',
               ha='center', fontsize=11, fontweight='bold', transform=ax[2].transAxes)

    flagged = p_std > 0.15
    banner = 'FLAGGED FOR ADDITIONAL REVIEW' if flagged else 'NOT FLAGGED'
    bcol   = '#E67E22' if flagged else '#7F8C8D'
    ax[2].text(0.5, 0.76, banner, ha='center', fontsize=11, fontweight='bold', color=bcol,
               transform=ax[2].transAxes,
               bbox=dict(boxstyle='round', facecolor='#FDEBD0' if flagged else '#ECF0F1',
                         edgecolor=bcol, lw=1.5))

    rows = [
        ('True label',            true_label),
        ('Ensemble probability',  f'{p_ens:.4f} -> {ens_lab}  ({"correct" if ens_ok else "ERROR"})'),
        ('DNN MC mean',           f'{p_mean:.4f} -> {mc_lab}  ({"correct" if mc_ok else "ERROR"})'),
        ('MC Dropout std',        f'{p_std:.4f}  (rank {rank_pos}/{len(mc_std)})'),
        ('Marginal conformal',    conformal_str),
    ]
    for k, (lab, val) in enumerate(rows):
        yy = 0.58 - k * 0.115
        ax[2].text(0.04, yy, lab + ':', fontsize=9, fontweight='bold', transform=ax[2].transAxes)
        ax[2].text(0.52, yy, val, fontsize=9, transform=ax[2].transAxes)

    if not ens_ok:
        note = 'Ensemble error; flagged by MC Dropout' if flagged else 'Ensemble error; not flagged'
    elif not mc_ok:
        note = 'Ensemble correct; the DNN alone would have erred'
    else:
        note = 'Both correct; retained for review on uncertainty alone'
    ax[2].text(0.5, 0.03, note, ha='center', fontsize=8, style='italic',
               color='gray', transform=ax[2].transAxes)

    plt.suptitle(f'TrustBreast | Objective 3 — Patient {PATIENT_IDX} case study',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTNAME, dpi=150, bbox_inches='tight')
    plt.show()

print("\nSaved: patient16_case.png  and  patient87_case.png")


## STEP 6c — ★ Sets for all 114: Mondrian composition, Patients 16/87, empty-set cross-tab

In [ ]:
# FIX E — conformal sets (marginal + Mondrian) for all 114, cross-tab, Patients 16/87
# File 3 — AFTER cell 19
# ============================================================================
import numpy as np
fmt = lambda s: '{' + ', '.join('M' if l == 1 else 'B' for l in s) + '}' if len(s) else '{} (empty)'
def comp(sets):
    z = [len(s) for s in sets]; return z.count(1), z.count(2), z.count(0)

y_arr = np.asarray(y_test).ravel().astype(int)
pred_ens = np.asarray(ens_pred).ravel().astype(int)
print("Set composition  (singleton / ambiguous / empty)")
print("  marginal :", comp(pred_sets))
print("  Mondrian :", comp(pred_sets_m))

for i in (16, 87):
    print(f"\nPatient {i}: true={'M' if y_arr[i] else 'B'} | marginal {fmt(pred_sets[i])} "
          f"| Mondrian {fmt(pred_sets_m[i])} | ensemble prob {float(np.ravel(prob_ensemble)[i]):.4f}")

empty_ids = [i for i, s in enumerate(pred_sets) if len(s) == 0]
flagged = set(np.where(mc_std > 0.15)[0].tolist())
errs    = set(np.where(pred_ens != y_arr)[0].tolist())
mc_errs = set(np.where((mc_mean >= .5).astype(int) != y_arr)[0].tolist())
print(f"\nEmpty sets (marginal): {empty_ids}")
print(f"  empty ∩ flagged : {sorted(set(empty_ids) & flagged)}  ({len(set(empty_ids) & flagged)}/{len(empty_ids)})")
print(f"  empty ∩ ensemble errors : {sorted(set(empty_ids) & errs)}")
print(f"  ensemble correct on all empty-set patients? {all(pred_ens[i] == y_arr[i] for i in empty_ids)}")
print(f"  flagged ({len(flagged)}) contains all ensemble errors {sorted(errs)}: {errs <= flagged}")
print(f"  flagged contains all MC-mean errors {sorted(mc_errs)}: {mc_errs <= flagged}")


## STEP 6d — ★ Split-conformal at α = 0.10 (reviewer B3, option 2)

In [ ]:
import numpy as np, math
def split_conformal(alpha):
    t,  _ = conformal_tau(cal_scores, alpha)
    tb, _ = conformal_tau(cal_scores[y_val_arr == 0], alpha)
    tm, _ = conformal_tau(cal_scores[y_val_arr == 1], alpha)
    inc_b, inc_m = test_mean <= t, (1 - test_mean) <= t
    cov  = np.mean(np.where(y_test_arr == 1, inc_m, inc_b))
    size = inc_b.astype(int) + inc_m.astype(int)
    cb = np.mean((test_mean <= tb)[y_test_arr == 0]); cm = np.mean(((1 - test_mean) <= tm)[y_test_arr == 1])
    nm = int((y_val_arr == 1).sum()); km = math.ceil((nm + 1) * (1 - alpha))
    print(f"alpha={alpha:.2f}: tau={t:.4f} marginal cov {cov*100:.2f}% | sets {np.sum(size==1)}/{np.sum(size==2)}/{np.sum(size==0)} (single/ambig/empty)")
    print(f"   Mondrian: tau_B={tb:.4f} cov_B {cb*100:.2f}% | tau_M={tm:.4f} cov_M {cm*100:.2f}%  "
          f"(malignant index {km}/{nm} -> {'SATURATED' if km >= nm else 'interior'})")
    return dict(tau=t, cov=cov, tb=tb, tm=tm, cb=cb, cm=cm, sets=(int(np.sum(size==1)), int(np.sum(size==2)), int(np.sum(size==0))))
SC = {a: split_conformal(a) for a in (0.05, 0.10)}


## STEP 7 — Final table + saved files (File 4 uses these files)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("=" * 60)
print("STEP 3.9 — FINAL HIGH-UNCERTAINTY CASE TABLE")
print("=" * 60)

# Define variables from previous cell's global scope
ece_score = ECE
mce_score = MCE
tau_fixed = tau
coverage_raw = marginal_cov # Assuming marginal_cov from prev cell is intended for coverage_raw

# Recalculate singleton_B, singleton_M, both_BM based on the current pred_sets (list of integer lists)
singleton_B = sum(1 for s in pred_sets if s == [0])
singleton_M = sum(1 for s in pred_sets if s == [1])
both_BM = sum(1 for s in pred_sets if s == [0, 1])

# ── 1. COMPREHENSIVE RESULTS TABLE ───────────────────────────
# Full summary for all patients
df_final = pd.DataFrame({
    'patient':    np.arange(len(y_test_arr)),
    'true':       ['M' if y==1 else 'B' for y in y_test_arr],
    'mc_mean':    mc_mean.round(4),
    'mc_std':     mc_std.round(4),
    'ci_lower':   mc_ci_lower.round(4),
    'ci_upper':   mc_ci_upper.round(4),
    'ci_width':   mc_ci_width.round(4),
    'pred_mc':    ['M' if p==1 else 'B' for p in mc_pred_class],
    'correct':    mc_pred_class == y_test_arr,
    'pred_set':   ['{'+','.join(map(str, s))+'}' for s in pred_sets], # FIXED: Convert integers to strings
    'unc_level':  ['High' if s>0.15 else
                   ('Medium' if s>0.05 else 'Low')
                   for s in mc_std],
    'biopsy':     mc_std > 0.15,
})

# ── 2. HIGH UNCERTAINTY TABLE (Paper Table) ───────────────────
print("\nTable: High-Uncertainty Patients (std > 0.15)")
print("=" * 72)
print(f"{'Pat':>4} {'True':>5} {'Pred':>5} {'Mean':>7} "
      f"{'Std':>7} {'CI Lower':>9} {'CI Upper':>9} "
      f"{'Set':>7} {'Risk':>20}")
print("-" * 72)

df_high_final = df_final[df_final['biopsy']].sort_values(
    'mc_std', ascending=False)

for _, r in df_high_final.iterrows():
    # Risk classification
    if r['true']=='M' and r['pred_mc']=='B':
        risk = '🔴 CRITICAL'
    elif r['true']=='M' and r['pred_mc']=='M':
        risk = '🟠 HIGH'
    else:
        risk = '🟡 MEDIUM'

    print(f"{int(r['patient']):>4} {r['true']:>5} "
          f"{r['pred_mc']:>5} {r['mc_mean']:>7.4f} "
          f"{r['mc_std']:>7.4f} {r['ci_lower']:>9.4f} "
          f"{r['ci_upper']:>9.4f} {r['pred_set']:>7}  "
          f"{risk}")

# ── 3. OBJECTIVE 3 COMPLETE SUMMARY ──────────────────────────
print()
print("=" * 60)
print("OBJECTIVE 3 — COMPLETE RESULTS SUMMARY")
print("=" * 60)

misclassified = df_final[~df_final['correct']]

print(f"""
── Step 3.1-3.3: MC Dropout (100 passes, n=114) ──
  Accuracy:              {(df_final['correct'].mean())*100:.2f}%
  Misclassified:         {len(misclassified)} patients
  Mean uncertainty:      {mc_std.mean():.4f}
  Max uncertainty:       {mc_std.max():.4f} (Patient {mc_std.argmax()})
  Mean 95% CI width:     {mc_ci_width.mean():.4f}

── Step 3.4: Uncertainty Triage ──
  Low    (std ≤ 0.05):  {(mc_std<=0.05).sum():3d} patients ({(mc_std<=0.05).mean()*100:.1f}%)
  Medium (0.05-0.15):   {((mc_std>0.05)&(mc_std<=0.15)).sum():3d} patients ({((mc_std>0.05)&(mc_std<=0.15)).mean()*100:.1f}%)
  High   (std > 0.15):  {(mc_std>0.15).sum():3d} patients ({(mc_std>0.15).mean()*100:.1f}%)

── Step 3.5-3.6: Conformal Prediction ──
  Calibration set:       91 held-out validation patients
  Threshold tau:         {tau_fixed:.4f}
  Empirical coverage:    {coverage*100:.2f}% (target: 95%)
  Singleton {{B}}:        {singleton_B} patients
  Singleton {{M}}:        {singleton_M} patients
  Ambiguous {{B,M}}:      {both_BM} patients (biopsy flag)

── Step 3.7-3.8: Calibration ──
  ECE:                   {ece_score:.4f} ({'well calibrated' if ece_score<0.05 else 'acceptable / moderately calibrated'})
  MCE:                   {mce_score:.4f}
  MCE note: inflated by a sparse (near-empty) confidence bin - interpret with caution

── Novel Findings for Paper ──
  1. MC-Dropout + split-conformal UQ on WBCD (no priority claimed)
  2. Most-uncertain patient (idx {mc_std.argmax()}, true {'M' if y_test_arr[mc_std.argmax()]==1 else 'B'}):
     {'correctly classified' if mc_pred_class[mc_std.argmax()]==y_test_arr[mc_std.argmax()] else 'MISclassified'} but flagged
     as highest-uncertainty (std={mc_std.max():.4f}) — clinical safety net
  3. {both_BM} patients: conformal set {{B,M}} — model
     explicitly admits uncertainty (not false confidence)
  4. Coverage {coverage*100:.2f}% (raw {coverage_raw*100:.2f}%) vs the 95% target
  5. ECE={ece_score:.4f} — descriptive only (no principled cut-off)
""")

# ── 4. FINAL VISUALIZATION ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# ── Plot 1: Complete uncertainty landscape ────────────────────
colors_unc = []
for i in range(len(y_test_arr)):
    if mc_std[i] > 0.15:
        colors_unc.append('#E74C3C')
    elif mc_std[i] > 0.05:
        colors_unc.append('#F39C12')
    else:
        colors_unc.append('#27AE60')

sc = axes[0][0].scatter(
    mc_mean, mc_std,
    c=colors_unc, s=30, alpha=0.8, zorder=3)

# Annotate most-uncertain patient (dynamic)
crit_idx = int(mc_std.argmax())
axes[0][0].annotate(
    f'P{crit_idx}\n[most uncertain]',
    (mc_mean[crit_idx], mc_std[crit_idx]),
    xytext=(mc_mean[crit_idx]+0.05,
            mc_std[crit_idx]+0.01),
    fontsize=8, color='darkred',
    arrowprops=dict(arrowstyle='->', color='darkred',
                    lw=1.2))

axes[0][0].axhline(0.15, color='red',
                   linestyle='--', lw=1.5,
                   label='Biopsy threshold (0.15)')
axes[0][0].axhline(0.05, color='orange',
                   linestyle='--', lw=1.5,
                   label='Medium threshold (0.05)')
axes[0][0].axvline(0.5, color='gray',
                   linestyle=':', lw=1, alpha=0.6)

red_p   = mpatches.Patch(color='#E74C3C',
                          label=f'High unc. ({(mc_std>0.15).sum()})')
ora_p   = mpatches.Patch(color='#F39C12',
                          label=f'Medium ({((mc_std>0.05)&(mc_std<=0.15)).sum()})')
grnc_p   = mpatches.Patch(color='#27AE60',
                          label=f'Low ({(mc_std<=0.05).sum()})')
axes[0][0].legend(handles=[red_p, ora_p, grnc_p],
                  fontsize=8)
axes[0][0].set_xlabel('MC Mean Probability')
axes[0][0].set_ylabel('MC Std (Uncertainty)')
axes[0][0].set_title('Uncertainty Landscape\n'
                     'Red=High, Orange=Medium, Green=Low',
                     fontweight='bold')
axes[0][0].grid(alpha=0.3)

# ── Plot 2: CI plot sorted ────────────────────────────────────
sort_idx = np.argsort(mc_mean)
pt_colors = ['#E74C3C' if y==1 else '#3498DB'
             for y in y_test_arr[sort_idx]]

axes[0][1].scatter(range(len(sort_idx)),
                   mc_mean[sort_idx],
                   c=pt_colors, s=20, zorder=3)
axes[0][1].fill_between(
    range(len(sort_idx)),
    mc_ci_lower[sort_idx],
    mc_ci_upper[sort_idx],
    alpha=0.15, color='gray', label='95% CI')
axes[0][1].axhline(
    0.5, color='black', linestyle='--', lw=1)
red_p2  = mpatches.Patch(color='#E74C3C',
                          label='Malignant')
blue_p2 = mpatches.Patch(color='#3498DB',
                          label='Benign')
axes[0][1].legend(handles=[red_p2, blue_p2],
                  fontsize=8)
axes[0][1].set_xlabel('Patients (sorted by MC mean)')
axes[0][1].set_ylabel('Probability')
axes[0][1].set_title('MC Mean ± 95% CI\n'
                     'Sorted by prediction confidence',
                     fontweight='bold')
axes[0][1].grid(alpha=0.3)

# ── Plot 3: Reliability diagram ───────────────────────────────
bin_edges   = np.linspace(0, 1, 11)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
valid       = ~np.isnan(bin_acc)

axes[1][0].plot([0,1],[0,1],'k--',lw=1.5,
                label='Perfect calibration')
axes[1][0].bar(bin_centers[valid],
               bin_acc[valid], width=0.08,
               alpha=0.7, color='steelblue',
               label='MC Dropout')
axes[1][0].bar(bin_centers[valid],
               np.maximum(
                   bin_conf[valid]-bin_acc[valid], 0),
               bottom=bin_acc[valid],
               width=0.08, alpha=0.3,
               color='red', label='Gap')
axes[1][0].set_xlabel('Confidence')
axes[1][0].set_ylabel('Accuracy')
axes[1][0].set_title(
    f'Reliability Diagram\nECE={ece_score:.4f}',
    fontweight='bold')
axes[1][0].legend(fontsize=8)
axes[1][0].grid(alpha=0.3)

# ── Plot 4: Conformal prediction sets ────────────────────────
labels_cp   = ['Confident\nBenign {B}',
               'Confident\nMalignant {M}',
               'Uncertain\n{B,M}']
counts_cp   = [singleton_B, singleton_M, both_BM]
colors_cp   = ['#3498DB', '#E74C3C', '#F39C12']
bars_cp     = axes[1][1].bar(
    labels_cp, counts_cp,
    color=colors_cp, edgecolor='white', width=0.5)

for bar, val in zip(bars_cp, counts_cp):
    pct = val / len(y_test_arr) * 100
    axes[1][1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        f'{val}\n({pct:.1f}%)',
        ha='center', fontsize=10,
        fontweight='bold')

axes[1][1].set_ylabel('Patients')
axes[1][1].set_title(
    f'Conformal Prediction Sets\n'
    f'Coverage={coverage*100:.2f}% | tau={tau_fixed:.4f}',
    fontweight='bold')
axes[1][1].set_ylim(0, max(counts_cp)*1.2)
axes[1][1].grid(axis='y', alpha=0.3)

plt.suptitle(
    'Objective 3 — Uncertainty Quantification Summary\n'
    f'MC Dropout (100 passes) + Conformal Prediction | '
    f'n=114 | Acc={df_final["correct"].mean()*100:.2f}%',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('objective3_complete.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("=" * 60)
print("✅ OBJECTIVE 3 — FULLY COMPLETE!")
print("=" * 60)
print("Saved: objective3_complete.png")
print()
print("All variables ready for O4 (DiCE) and O6 (LLM):")
print(f"  mc_mean      — shape {mc_mean.shape}")
print(f"  mc_std       — shape {mc_std.shape}")
print(f"  mc_ci_lower  — shape {mc_ci_lower.shape}")
print(f"  mc_ci_upper  — shape {mc_ci_upper.shape}")
print(f"  pred_sets    — list of {len(pred_sets)}")
print(f"  df_mc        — DataFrame {df_mc.shape}")
print(f"  ece_score    = {ece_score:.4f}")
print(f"  coverage     = {coverage*100:.2f}%")

In [ ]:
# ============================================================
# OBJECTIVE 3 — SAVE ALL RESULTS TO FILES
# ============================================================
import numpy as np
import pandas as pd
import json
from datetime import datetime

print("=" * 55)
print("SAVING OBJECTIVE 3 RESULTS")
print("=" * 55)

# ── 1. MAIN RESULTS CSV ──────────────────────────────────────
df_save = pd.DataFrame({
    'patient_idx':  np.arange(len(y_test_arr)),
    'true_label':   ['M' if y==1 else 'B' for y in y_test_arr],
    'mc_mean':      mc_mean.round(4),
    'mc_std':       mc_std.round(4),
    'ci_lower':     mc_ci_lower.round(4),
    'ci_upper':     mc_ci_upper.round(4),
    'ci_width':     mc_ci_width.round(4),
    'pred_class':   ['M' if p==1 else 'B' for p in mc_pred_class],
    'correct':      mc_pred_class == y_test_arr,
    'pred_set':     ['{'+','.join(map(str, s))+'}' for s in pred_sets],
    'unc_level':    ['High' if s>0.15 else
                     ('Medium' if s>0.05 else 'Low')
                     for s in mc_std],
    'biopsy_flag':  mc_std > 0.15,
})

df_save.to_csv('O3_all_patients.csv', index=False)
print(f"✅ O3_all_patients.csv — {len(df_save)} rows")

# ── 2. HIGH UNCERTAINTY TABLE CSV ────────────────────────────
df_high_save = df_save[df_save['biopsy_flag']].copy()
df_high_save = df_high_save.sort_values(
    'mc_std', ascending=False)

def risk_label(row):
    if row['true_label']=='M' and row['pred_class']=='B':
        return 'CRITICAL — Missed malignant'
    elif row['true_label']=='M' and row['pred_class']=='M':
        return 'HIGH — Malignant uncertain'
    else:
        return 'MEDIUM — Benign borderline'

df_high_save['clinical_risk'] = df_high_save.apply(
    risk_label, axis=1)
df_high_save.to_csv('O3_high_uncertainty.csv', index=False)
print(f"✅ O3_high_uncertainty.csv — {len(df_high_save)} rows")

# ── 3. MC PROBS MATRIX (all 100 passes) ──────────────────────
np.save('O3_mc_probs_100passes.npy', mc_probs)
print(f"✅ O3_mc_probs_100passes.npy — shape {mc_probs.shape}")

# ── 4. CONFORMAL PREDICTION CSV ──────────────────────────────
df_conformal = pd.DataFrame({
    'patient_idx': np.arange(len(y_test_arr)),
    'true_label':  ['M' if y==1 else 'B' for y in y_test_arr],
    'mc_mean':     mc_mean.round(4),
    'mc_std':      mc_std.round(4),
    'pred_set':    ['{'+','.join(map(str, s))+'}' for s in pred_sets],
    'set_size':    [len(s) for s in pred_sets],
    'covered':     covered,
    'ambiguous':   [(0 in s) and (1 in s) for s in pred_sets],
})
df_conformal.to_csv('O3_conformal_prediction.csv', index=False)
print(f"✅ O3_conformal_prediction.csv — {len(df_conformal)} rows")

# ── 5. SUMMARY METRICS JSON ──────────────────────────────────
summary = {
    'objective': 'O3 — Uncertainty Quantification',
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'dataset': {
        'test_patients': int(len(y_test_arr)),
        'benign':        int((y_test_arr==0).sum()),
        'malignant':     int((y_test_arr==1).sum()),
    },
    'mc_dropout': {
        'n_passes':        100,
        'accuracy':        round(float(
            (mc_pred_class==y_test_arr).mean()*100), 2),
        'misclassified':   int(
            (mc_pred_class!=y_test_arr).sum()),
        'mean_std':        round(float(mc_std.mean()), 4),
        'max_std':         round(float(mc_std.max()),  4),
        'mean_ci_width':   round(float(
            mc_ci_width.mean()), 4),
        'critical_patient': int(mc_std.argmax()),
    },
    'uncertainty_triage': {
        'low_n':      int((mc_std<=0.05).sum()),
        'low_pct':    round(float(
            (mc_std<=0.05).mean()*100), 1),
        'medium_n':   int(((mc_std>0.05)&
                           (mc_std<=0.15)).sum()),
        'medium_pct': round(float(((mc_std>0.05)&
                           (mc_std<=0.15)).mean()*100), 1),
        'high_n':     int((mc_std>0.15).sum()),
        'high_pct':   round(float(
            (mc_std>0.15).mean()*100), 1),
    },
    'conformal_prediction': {
        'alpha':            0.05,
        'n_calibration':    91,
        'tau':              round(float(tau_fixed), 4),
        'coverage_raw':     round(float(coverage_raw*100), 2),
        'coverage_reported':round(float(coverage*100), 2),
        'coverage_note':    'empty sets NOT padded; reported = raw conformal coverage',
        'singleton_B':      int(singleton_B),
        'singleton_M':      int(singleton_M),
        'ambiguous_BM':     int(both_BM),
    },
    'calibration': {
        'ECE': round(float(ece_score), 4),
        'MCE': round(float(mce_score), 4),
        'n_bins': 10,
        'interpretation': ('Well calibrated (ECE < 0.05)'
                           if ece_score < 0.05
                           else 'Acceptable / moderately calibrated (ECE >= 0.05); '
                                'MCE inflated by a sparse confidence bin'),
    },
    'novel_findings': [
        'MC-Dropout + split-conformal UQ on WBCD (no priority claimed)',
        f'Most-uncertain patient (idx {int(mc_std.argmax())}, '
        f"true {'M' if y_test_arr[int(mc_std.argmax())]==1 else 'B'}): "
        f"{'correctly classified' if mc_pred_class[int(mc_std.argmax())]==y_test_arr[int(mc_std.argmax())] else 'misclassified'} "
        f'but flagged as highest-uncertainty (std={mc_std.max():.4f})',
        f'{both_BM} patients received ambiguous {{B,M}} '
        f'conformal sets',
        f'Reported coverage {round(float(coverage*100),2)}% '
        f'(raw {round(float(coverage_raw*100),2)}%) vs the 95% target',
        f'ECE={round(float(ece_score),4)} — '
        f"descriptive calibration statistic",
    ]
}

with open('O3_summary_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ O3_summary_metrics.json")

# ── 6. RELIABILITY DIAGRAM DATA CSV ──────────────────────────
bin_edges_save = np.linspace(0, 1, 11)
df_reliability = pd.DataFrame({
    'bin_lower':  bin_edges_save[:-1].round(1),
    'bin_upper':  bin_edges_save[1:].round(1),
    'n_patients': bin_n,
    'mean_conf':  np.where(bin_n>0,
                           bin_conf, np.nan).round(4),
    'mean_acc':   np.where(bin_n>0,
                           bin_acc, np.nan).round(4),
    'gap':        np.where(bin_n>0,
                           np.abs(bin_acc-bin_conf),
                           np.nan).round(4),
})
df_reliability.to_csv('O3_reliability_diagram.csv',
                      index=False)
print(f"✅ O3_reliability_diagram.csv")

# ── 7. PRINT ALL FILES ───────────────────────────────────────
print()
print("=" * 55)
print("ALL FILES SAVED")
print("=" * 55)
files = [
    ('O3_all_patients.csv',          'All 114 patients full results'),
    ('O3_high_uncertainty.csv',      f'{int((mc_std>0.15).sum())} high-uncertainty patients'),
    ('O3_mc_probs_100passes.npy',    '100-pass MC matrix (114×100)'),
    ('O3_conformal_prediction.csv',  'Conformal prediction sets'),
    ('O3_summary_metrics.json',      'All metrics for paper'),
    ('O3_reliability_diagram.csv',   'Reliability diagram data'),
    ('mc_dropout_uncertainty.png',   'MC Dropout plots'),
    ('reliability_calibration.png',  'Reliability + ECE plots'),
    ('objective3_complete.png',      'Full O3 summary plot'),
]

for fname, desc in files:
    print(f"  📄 {fname}")
    print(f"     → {desc}")

print()
print("Download from Colab:")
print("  Files panel (left) → right-click → Download")
print("  Or run:")
print("  from google.colab import files")
print("  files.download('O3_summary_metrics.json')")
print()
print("=" * 55)
print("✅ OBJECTIVE 3 — 100% COMPLETE!")
print("=" * 55)
print()
print("Next: Objective 4 — DiCE Counterfactuals!")


## STEP 8 — ★ RESULTS SUMMARY + reproducibility check (share only this cell's output)

In [ ]:
import numpy as np
fmt = lambda s: '{' + ', '.join('M' if l == 1 else 'B' for l in s) + '}' if len(s) else '{}'
rank = {int(ix): r + 1 for r, ix in enumerate(np.argsort(-mc_std))}
p_ens = np.asarray(prob_ensemble).ravel()
now = {
 "MC accuracy %":           round(float(np.mean((mc_mean > .5).astype(int) == y_test_arr))*100, 2),
 "mean std":                round(float(mc_std.mean()), 4),
 "mean 95% width":          round(float(mc_ci_width.mean()), 4),
 "flagged (std>0.15)":      sorted(np.where(mc_std > 0.15)[0].tolist()),
 "MC-mean errors":          sorted(np.where((mc_mean >= .5).astype(int) != y_test_arr)[0].tolist()),
 "P16 std / rank":          (round(float(mc_std[16]), 4), rank[16]),
 "P87 std / rank":          (round(float(mc_std[87]), 4), rank[87]),
 "P16 ensemble prob":       round(float(p_ens[16]), 4),
 "tau marginal":            round(tau, 4),
 "marginal coverage %":     round(float(marginal_cov)*100, 2),
 "sets single/ambig/empty": (singleton, both, empty),
 "tau_B / tau_M":           (round(tau_ben, 4), round(tau_mal, 4)),
 "Mondrian cov B / M %":    (round(cov_ben*100, 2), round(cov_mal*100, 2)),
 "ECE / MCE":               (round(ECE, 4), round(MCE, 4)),
 "empty-set patients":      sorted([i for i, s in enumerate(pred_sets) if len(s) == 0]),
}
before = {
 "MC accuracy %": 98.25, "mean std": 0.0531, "mean 95% width": 0.1815,
 "flagged (std>0.15)": [3, 16, 18, 29, 42, 56, 61, 62, 79, 87, 111, 112],
 "MC-mean errors": [16, 87], "P16 std / rank": (0.1865, 9), "P87 std / rank": (0.1808, 10),
 "P16 ensemble prob": 0.3348, "tau marginal": 0.3511, "marginal coverage %": 93.86,
 "sets single/ambig/empty": (109, 0, 5), "tau_B / tau_M": (0.3267, 0.9225),
 "Mondrian cov B / M %": (97.22, 100.0), "ECE / MCE": (0.0474, 0.4273),
 "empty-set patients": [3, 18, 29, 79, 112],
}
print("="*72 + "\nREPRODUCIBILITY CHECK (vs. previous File 3 run)\n" + "="*72)
ok_all = True
for k in before:
    ok = (now[k] == before[k]) or (isinstance(before[k], float) and abs(now[k] - before[k]) < 5e-4)
    ok_all &= ok
    print(f"  {'✅' if ok else '❌'} {k:<26} now {now[k]}   before {before[k]}")
print("\nALL MATCH ✅ — File 3 is reproducible" if ok_all else "\nSome numbers changed ❌ — share the output")

print("\n" + "="*72 + "\nFOR PAPER CORRECTIONS (new facts)\n" + "="*72)
for i in (16, 87):
    print(f"  Patient {i}: marginal {fmt(pred_sets[i])} | Mondrian {fmt(pred_sets_m[i])}")
z = [len(s) for s in pred_sets_m]
print(f"  Mondrian sets (single/ambig/empty): {z.count(1)}/{z.count(2)}/{z.count(0)}")
e = [i for i, s in enumerate(pred_sets) if len(s) == 0]
print(f"  Empty sets all flagged: {set(e) <= set(np.where(mc_std > 0.15)[0])} | "
      f"all classified correctly by ensemble: {all(int(np.ravel(ens_pred)[i]) == y_test_arr[i] for i in e)}")
for a, r in SC.items():
    print(f"  alpha={a:.2f}: marginal {r['cov']*100:.2f}% (tau {r['tau']:.4f}, sets {r['sets']}) | "
          f"Mondrian B {r['cb']*100:.2f}% (tau {r['tb']:.4f}), M {r['cm']*100:.2f}% (tau {r['tm']:.4f})")


## STEP 9 — Download figures + CSVs as one zip

In [ ]:
import glob, zipfile
files_ = sorted(set(glob.glob('*.png') + glob.glob('O3_*.csv') + glob.glob('O3_*.json') + glob.glob('O3_*.npy')))
with zipfile.ZipFile('File3_results.zip', 'w') as z:
    for f in files_: z.write(f)
print(f"{len(files_)} files zipped:"); [print("  ", f) for f in files_]
print("\nFor the paper: Fig 9 -> mc_dropout_uncertainty.png | Fig 10 -> O3_reliability_conformal_FIXED.png"
      "\n               Fig 11 -> patient16_case.png | Fig 12 -> patient87_case.png")
try:
    from google.colab import files; files.download('File3_results.zip')
except Exception:
    print("Download File3_results.zip from the Files panel on the left.")
